In [14]:
import cv2
import numpy as np
from __future__ import print_function
import math

In [16]:
cap=cv2.VideoCapture("Volleyball.mp4")


# backSub = cv2.createBackgroundSubtractorMOG2()

backSub = cv2.createBackgroundSubtractorKNN()

while(True):
    ret,frame=cap.read()
    #To convert BGR to HSV
    hsv=cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)
    print(frame.shape)

    # edge=cv2.Canny(frame,50,150)



    lower_yellow = np.array([15, 150, 160])
    upper_yellow = np.array([22, 255, 255])

    lower_blue = np.array([110, 50, 50])
    upper_blue = np.array([130, 255, 255])
    #Creating mask for the ball
    mask=cv2.inRange(hsv,lower_yellow,upper_yellow)
    #Creating mask for the argentinian players
    mask_p=cv2.inRange(hsv,lower_blue,upper_blue)

    height, width, channels = frame.shape



    contours, hierarchy = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        min_contour = max(contours, key=cv2.contourArea)

    best_contour = None
    best_score = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if 100<area<350:  # filter size as ball area is small
         continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        aspectRatio = float(w)/h

        if y<height*0.25:
            min_r,max_r=4,13
        elif y<height*0.65:
            min_r,max_r = 8,15
        else:
            min_r,max_r=7,17

        radius = max(w,h)/2
        if not (min_r<radius<max_r):
            continue

        circularity = 4 * math.pi * area / (perimeter * perimeter)
        if 0.7<circularity<1.2 and 1<aspectRatio<1.2:  # strong circle filter
                 best_score = circularity
                 best_contour = cnt

        if best_contour is not None:
                     x, y, w, h = cv2.boundingRect(best_contour)
                     cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

        #For motion tracking , as the ball moves much faster than the players

        prev_center = None
        cx, cy = int(width/2), int(height/2)
        if best_contour is not None:
            cx = (x + w) // 2
            cy = (y + h) // 2

        if prev_center is not None:
            dist = np.linalg.norm(np.array([cx, cy]) - np.array(prev_center))

            if dist < 10:
                best_contour = None
            else:
                prev_center = (cx, cy)

        else:
            prev_center = (cx, cy)


    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7, 7))
    mask=cv2.GaussianBlur(mask,(7,7),0)
    #To remove noise(erosion followed by dilation)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    #To remove small holes in the objects(dilation followed by erosion)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    res = cv2.bitwise_and(hsv,frame,mask=mask)
    # fgMask = backSub.apply(frame)



    cv2.putText(frame, f"Ball ({cx},{cy})",
                        (x, y-10), cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (255, 0, 0), 2)



    # cv2.imshow('Frame', frame)
    # cv2.imshow('FG Mask', fgMask)

    cnt_p,h_p=cv2.findContours(mask_p,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7, 7))
    mask_p=cv2.GaussianBlur(mask_p,(7,7),0)
    #To remove noise(erosion followed by dilation)
    mask_p = cv2.morphologyEx(mask_p, cv2.MORPH_OPEN, kernel)
    #To remove small holes in the objects(dilation followed by erosion)
    mask_p = cv2.morphologyEx(mask_p, cv2.MORPH_CLOSE, kernel)


    #For creating the bounding rectangle for the argentinian players
    for i in cnt_p:
        area = cv2.contourArea(i)
        x_p,y_p,w_p,h_p = cv2.boundingRect(i)
        if 1500<area<2000:
            cv2.rectangle(frame,(x_p,y_p),(x_p+w_p,y_p+h_p),(0,255,0),2)
            cv2.putText(frame,f"ARG ({x_p},{y_p})",(x_p,y_p-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

    mask_y=cv2.inRange(hsv,lower_yellow,upper_yellow)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7, 7))
    mask_y=cv2.GaussianBlur(mask_y,(7,7),0)
    #To remove noise(erosion followed by dilation)
    mask_y = cv2.morphologyEx(mask_y, cv2.MORPH_OPEN, kernel)
    #To remove small holes in the objects(dilation followed by erosion
    mask_y = cv2.morphologyEx(mask_y, cv2.MORPH_CLOSE, kernel)
    cnt_y,h_y=cv2.findContours(mask_y,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    #For creating the bounding rectangle for the brazilian players
    for i in cnt_y:
        area = cv2.contourArea(i)
        x_y,y_y,w_y,h_y = cv2.boundingRect(i)
        aspect_ratio = float(w_y)/h_y
        if 1500<area<2000:
            cv2.rectangle(frame,(x_y,y_y),(x_y+w_y,y_y+h_y),(128,0,128),2)
            cv2.putText(frame,f"BRA ({x_y},{y_y})",(x_y,y_y-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(128,128,128),2)





    cv2.imshow('frame',frame)
    cv2.imshow('mask',mask)
    cv2.imshow('res',res)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280, 3)
(720, 1280